<div dir="rtl" align="right">

# پس‌انتشار خطا با عدد واقعی

درسی از مدرسه پایتون و هوش مصنوعی، دکتر بهروز طاهری

## شبکه ما
یک شبکه کوچک: 2 ورودی، 2 نورون پنهان با سیگموید، 1 خروجی با سیگموید و خطای $L=\frac12(\hat{y}-y)^2$. برای ساده بودن بایاس نداریم.

> [شبکه 2-2-1 با همه مقادیر گذر رفت (داخل گره‌ها) و خطاهای δ در گذر برگشت (سبز).](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/math_backprop.png)

*شبکه 2-2-1 با همه مقادیر گذر رفت (داخل گره‌ها) و خطاهای δ در گذر برگشت (سبز).*

## گام 1: گذر رفت (forward)
| کمیت | فرمول | محاسبه | مقدار |
|---|---|---|---|
| $z_1$ | $w_{11}x_1+w_{12}x_2$ | $0.1\times1+0.2\times2$ | 0.5 |
| $h_1$ | $\sigma(z_1)$ | | 0.622 |
| $z_2$ | $w_{21}x_1+w_{22}x_2$ | $0.3\times1+0.4\times2$ | 1.1 |
| $h_2$ | $\sigma(z_2)$ | | 0.750 |
| $z_o$ | $v_1h_1+v_2h_2$ | $0.5\times0.622+0.6\times0.750$ | 0.761 |
| $\hat{y}$ | $\sigma(z_o)$ | | 0.682 |
| $L$ | $\frac12(\hat{y}-y)^2$ | $\frac12(0.682-1)^2$ | 0.0507 |

## گام 2: خطای لایه خروجی
«خطای» هر نورون را $\delta=\partial L/\partial z$ تعریف می‌کنیم:

$$\delta_o=(\hat{y}-y)\cdot\hat{y}(1-\hat{y})=(-0.318)(0.682)(0.318)=-0.0691$$

## گام 3: برگرداندن خطا به لایه پنهان
هر نورون پنهان به اندازه **وزنی که به خروجی وصلش می‌کند** در خطا سهم دارد:

$$\delta_j=\delta_o\cdot v_j\cdot h_j(1-h_j)$$

| نورون | محاسبه | مقدار |
|---|---|---|
| $\delta_1$ | $-0.0691\times0.5\times0.622\times0.378$ | −0.0081 |
| $\delta_2$ | $-0.0691\times0.6\times0.750\times0.250$ | −0.0078 |

## گام 4: گرادیان هر وزن
قاعده طلایی: **گرادیان وزن = خطای نورون مقصد × خروجی نورون مبدأ**.

| وزن | فرمول | مقدار گرادیان | وزن جدید با $\eta=0.5$ |
|---|---|---|---|
| $v_1$ | $\delta_o h_1$ | −0.0430 | 0.5215 |
| $v_2$ | $\delta_o h_2$ | −0.0518 | 0.6259 |
| $w_{11}$ | $\delta_1 x_1$ | −0.0081 | 0.1041 |
| $w_{12}$ | $\delta_1 x_2$ | −0.0162 | 0.2081 |
| $w_{21}$ | $\delta_2 x_1$ | −0.0078 | 0.3039 |
| $w_{22}$ | $\delta_2 x_2$ | −0.0155 | 0.4078 |

همه گرادیان‌ها منفی‌اند؛ پس همه وزن‌ها کمی زیاد می‌شوند تا خروجی از 0.682 به سمت 1 برود. بعد از این یک گام، خطا از 0.0507 به حدود 0.0481 می‌رسد.

## فرمول کلی به زبان ماتریس
برای لایه $l$ با ورودی $a^{(l-1)}$، وزن $W^{(l)}$ و فعال‌سازی f:

| مرحله | فرمول |
|---|---|
| رفت | $z^{(l)}=W^{(l)}a^{(l-1)}+b^{(l)},\quad a^{(l)}=f(z^{(l)})$ |
| خطای لایه آخر | $\delta^{(L)}=\nabla_a L\odot f'(z^{(L)})$ |
| خطای لایه‌های قبل | $\delta^{(l)}=\big(W^{(l+1)\top}\delta^{(l+1)}\big)\odot f'(z^{(l)})$ |
| گرادیان وزن | $\partial L/\partial W^{(l)}=\delta^{(l)}\,a^{(l-1)\top}$ |
| گرادیان بایاس | $\partial L/\partial b^{(l)}=\delta^{(l)}$ |

$\odot$ ضرب درایه به درایه است. توجه کن که در برگشت از **ترانهاده** همان ماتریس وزنِ گذر رفت استفاده می‌شود.

## چرا گرادیان محو یا منفجر می‌شود؟
در هر لایه، δ در $W^\top$ و در $f'(z)$ ضرب می‌شود. بعد از L لایه، L بار ضرب شده است:

| وضعیت | نتیجه | راه‌حل |
|---|---|---|
| $f'$ کوچک (سیگموید: حداکثر 0.25) | گرادیان محو می‌شود | ReLU، اتصال میان‌بر (ResNet)، نرمال‌سازی دسته‌ای |
| وزن‌های بزرگ | گرادیان منفجر می‌شود | بریدن گرادیان (clipping)، مقداردهی اولیه درست |
| وزن‌های اولیه همه صفر | همه نورون‌ها یکسان می‌مانند | مقداردهی تصادفی |

| مقداردهی اولیه | واریانس وزن‌ها | مناسب برای |
|---|---|---|
| Xavier / Glorot | $\dfrac{2}{n_{in}+n_{out}}$ | سیگموید، tanh |
| He | $\dfrac{2}{n_{in}}$ | ReLU |

## بررسی گرادیان
همیشه می‌توان پس‌انتشار را با مشتق عددی بررسی کرد: یک وزن را کمی تغییر بده و ببین خطا چقدر عوض می‌شود. کد زیر همین کار را می‌کند و سپس شبکه را برای حل مسئله XOR آموزش می‌دهد.



</div>

In [ ]:
import numpy as np

sig = lambda z: 1 / (1 + np.exp(-z))

x = np.array([1.0, 2.0]); y = 1.0
W = np.array([[0.1, 0.2],        # weights into hidden neuron 1
              [0.3, 0.4]])       # weights into hidden neuron 2
v = np.array([0.5, 0.6])         # hidden -> output

def forward(W, v):
    h = sig(W @ x); y_hat = sig(v @ h)
    return h, y_hat, 0.5 * (y_hat - y) ** 2

h, y_hat, L = forward(W, v)
print("h =", h.round(4), "  y_hat =", round(y_hat, 4), "  L =", round(L, 4))

delta_o = (y_hat - y) * y_hat * (1 - y_hat)
delta_h = delta_o * v * h * (1 - h)
grad_v = delta_o * h
grad_W = np.outer(delta_h, x)
print("delta_o =", round(delta_o, 4), "  delta_h =", delta_h.round(4))
print("grad_v =", grad_v.round(4)); print("grad_W ="); print(grad_W.round(4))

# Gradient check for w11
eps = 1e-6; Wp = W.copy(); Wp[0, 0] += eps; Wm = W.copy(); Wm[0, 0] -= eps
print("numeric dL/dw11 =", round((forward(Wp, v)[2] - forward(Wm, v)[2]) / (2 * eps), 6), "  backprop =", round(grad_W[0, 0], 6))

W2, v2 = W - 0.5 * grad_W, v - 0.5 * grad_v
print("loss after one step:", round(forward(W2, v2)[2], 4))

# Train a 2-4-1 network on XOR with the matrix formulas
rng = np.random.default_rng(1)
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float); T = np.array([[0], [1], [1], [0]], dtype=float)
W1 = rng.normal(0, 1, (2, 4)); b1 = np.zeros(4); W2 = rng.normal(0, 1, (4, 1)); b2 = np.zeros(1)
for epoch in range(1, 5001):
    H = sig(X @ W1 + b1); Y = sig(H @ W2 + b2)                  # forward
    d2 = (Y - T) * Y * (1 - Y); d1 = (d2 @ W2.T) * H * (1 - H)  # backward
    W2 -= 1.0 * H.T @ d2; b2 -= 1.0 * d2.sum(0); W1 -= 1.0 * X.T @ d1; b1 -= 1.0 * d1.sum(0)
    if epoch % 1000 == 0: print(f"epoch {epoch}: loss = {0.5 * np.sum((Y - T) ** 2):.4f}")
print("XOR predictions:", Y.ravel().round(2))

<div dir="rtl" align="right">

### تمرین

در همین شبکه اگر $y=0$ بود، $\delta_o$ و گرادیان $v_1$ چه می‌شد؟ علامت‌ها چه چیزی را نشان می‌دهند؟

</div>

In [ ]:
# Your answer here